In [90]:
import json
import os
import pandas as pd
from transformers import AutoTokenizer
import re
import json
from dotenv import load_dotenv
from glob import glob
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))

In [91]:
from src.utils.eval_utils import calculate_metrics

In [92]:
seed = '123'
dataset_type = 'hoasa_hotel_as'
lang = 'indo'
results_paths = glob(f'outputs/evals/{dataset_type}/{lang}/*/seed_{seed}/*/*/*/*/inference_results.json')

In [93]:
results_paths

['outputs/evals/hoasa_hotel_as/indo/gas/seed_123/20251226_133717_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-2800/checkpoint-2800/constrained_decoding/inference_results.json',
 'outputs/evals/hoasa_hotel_as/indo/gas/seed_123/20251226_133717_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-2800/checkpoint-2800/unconstrained_decoding/inference_results.json',
 'outputs/evals/hoasa_hotel_as/indo/mvp/seed_123/20251226_140621_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-5590/checkpoint-5590/constrained_decoding/inference_results.json',
 'outputs/evals/hoasa_hotel_as/indo/mvp/seed_123/20251226_140621_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-5590/checkpoint-5590/unconstrained_decoding/inference_results.json',
 'outputs/evals/hoasa_hotel_as/indo/mvp_aos/seed_123/20251226_125838_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-2800/checkpoint-2800/constrained_decoding/inference_results.json',
 'outputs/evals/hoasa_hotel_as/i

In [94]:
results_dict = {}
for path in results_paths:
	with open(path, 'r') as f:
		results = json.load(f)
	dataset_folder = path.split('/')[4]
	decoding = path.split('/')[-2]
	results_dict[f'{dataset_folder}-{decoding}'] = results

In [95]:
df_dict = {}
for key, results in results_dict.items():
	df_dict[key] = pd.DataFrame(results)
	df_dict[key]['metrics'] = df_dict[key].apply(lambda row: calculate_metrics([row['target_list']], [row['prediction_list']], task='aos'), axis=1)
	df_dict[key]['f1_score'] = df_dict[key]['metrics'].apply(lambda x: x['f1_aos'])
	df_dict[key]['target_list'] = df_dict[key]['target_list'].apply(lambda x: '\n--------\n'.join(x))
	df_dict[key]['prediction_list'] = df_dict[key]['prediction_list'].apply(lambda x: '\n--------\n'.join(x))

In [96]:
df_dict.keys()

dict_keys(['gas-constrained_decoding', 'gas-unconstrained_decoding', 'mvp-constrained_decoding', 'mvp-unconstrained_decoding', 'mvp_aos-constrained_decoding', 'mvp_aos-unconstrained_decoding', 'indolegoabsa_multitask-constrained_decoding', 'indolegoabsa_multitask-unconstrained_decoding'])

In [97]:
new_df = df_dict['mvp_aos-constrained_decoding'].copy()
for key, df in df_dict.items():
	if key == 'mvp_aos-constrained_decoding':
		continue
	dataset_folder = key.split('-')[0]
	decoding = key.split('-')[1]

	key = key.replace('_old', '')

	new_df = new_df.merge(df, on='sentence_id', suffixes=('', f'-{key}')).copy()
	

In [98]:
list(df_dict.keys())

['gas-constrained_decoding',
 'gas-unconstrained_decoding',
 'mvp-constrained_decoding',
 'mvp-unconstrained_decoding',
 'mvp_aos-constrained_decoding',
 'mvp_aos-unconstrained_decoding',
 'indolegoabsa_multitask-constrained_decoding',
 'indolegoabsa_multitask-unconstrained_decoding']

In [99]:
selected_columns = ['sentence_id', 'element_order', 'input', 'target_list', 'prediction_list', 'f1_score', 'prediction_list-mvp_aos-unconstrained_decoding', 'f1_score-mvp_aos-unconstrained_decoding']
for key in df_dict.keys():
	if 'mvp_aos-' in key:
		print('Skipping', key)
		continue
	key = key.replace('_old', '')
	selected_columns.append(f'prediction_list-{key}')
	selected_columns.append(f'f1_score-{key}')
selected_columns

Skipping mvp_aos-constrained_decoding
Skipping mvp_aos-unconstrained_decoding


['sentence_id',
 'element_order',
 'input',
 'target_list',
 'prediction_list',
 'f1_score',
 'prediction_list-mvp_aos-unconstrained_decoding',
 'f1_score-mvp_aos-unconstrained_decoding',
 'prediction_list-gas-constrained_decoding',
 'f1_score-gas-constrained_decoding',
 'prediction_list-gas-unconstrained_decoding',
 'f1_score-gas-unconstrained_decoding',
 'prediction_list-mvp-constrained_decoding',
 'f1_score-mvp-constrained_decoding',
 'prediction_list-mvp-unconstrained_decoding',
 'f1_score-mvp-unconstrained_decoding',
 'prediction_list-indolegoabsa_multitask-constrained_decoding',
 'f1_score-indolegoabsa_multitask-constrained_decoding',
 'prediction_list-indolegoabsa_multitask-unconstrained_decoding',
 'f1_score-indolegoabsa_multitask-unconstrained_decoding']

In [100]:
for col in new_df.columns:
	print(col)

sentence_id
task_elements
element_order
input
target
prediction
target_list
prediction_list
metrics
f1_score
task_elements-gas-constrained_decoding
element_order-gas-constrained_decoding
input-gas-constrained_decoding
target-gas-constrained_decoding
prediction-gas-constrained_decoding
target_list-gas-constrained_decoding
prediction_list-gas-constrained_decoding
metrics-gas-constrained_decoding
f1_score-gas-constrained_decoding
task_elements-gas-unconstrained_decoding
element_order-gas-unconstrained_decoding
input-gas-unconstrained_decoding
target-gas-unconstrained_decoding
prediction-gas-unconstrained_decoding
target_list-gas-unconstrained_decoding
prediction_list-gas-unconstrained_decoding
metrics-gas-unconstrained_decoding
f1_score-gas-unconstrained_decoding
task_elements-mvp-constrained_decoding
element_order-mvp-constrained_decoding
input-mvp-constrained_decoding
target-mvp-constrained_decoding
prediction-mvp-constrained_decoding
target_list-mvp-constrained_decoding
prediction_list

In [101]:
'prediction_list-mvp_aos-constrained_decoding' in new_df.columns

False

In [102]:
new_df[selected_columns]

,sentence_id,element_order,input,target_list,prediction_list,f1_score,prediction_list-mvp_aos-unconstrained_decoding,f1_score-mvp_aos-unconstrained_decoding,prediction_list-gas-constrained_decoding,f1_score-gas-constrained_decoding,prediction_list-gas-unconstrained_decoding,f1_score-gas-unconstrained_decoding,prediction_list-mvp-constrained_decoding,f1_score-mvp-constrained_decoding,prediction_list-mvp-unconstrained_decoding,f1_score-mvp-unconstrained_decoding,prediction_list-indolegoabsa_multitask-constrained_decoding,f1_score-indolegoabsa_multitask-constrained_decoding,prediction_list-indolegoabsa_multitask-unconstrained_decoding,f1_score-indolegoabsa_multitask-unconstrained_decoding
0,3500,as,pelayanan nya sangat ramah . [A] [S] =>,[A] pelayanan nya [S] positive,[A] pelayanan nya [S] positive,1.000000,[A] pelayanan nya [S] positive,1.000000,( pelayanan nya | positive ),1.000000,( pelayanan nya | positive ),1.000000,[S] positive [A] pelayanan nya,0.0,[S] positive [A] pelayanan nya,0.0,<|aspect|> pelayanan nya <|sentiment|> positive,1.000000,<|aspect|> pelayanan nya <|sentiment|> positive,1.000000
1,3501,as,sayang wifi tidak bagus harus keluar kamar . [...,[A] wifi [S] negative,[A] wifi [S] negative,1.000000,[A] wifi [S] negative,1.000000,( wifi | negative ),1.000000,( wifi | negative ),1.000000,[A] wifi [S] negative,1.0,[A] wifi [S] negative,1.0,<|aspect|> wifi <|sentiment|> negative,1.000000,<|aspect|> wifi <|sentiment|> negative,1.000000
2,3502,as,"tulisannya twin bed , tetapi yang ada kamarnya...",[A] kamarnya [S] negative,[A] twin bed [S] negative\n--------\n[A] kamar...,0.666667,[A] twin bed [S] negative\n--------\n[A] kamar...,0.666667,( twin bed | positive )\n--------\n( kamarnya ...,0.666667,( twin bed | positive )\n--------\n( kamarnya ...,0.666667,[A] twin bed [S] positive\n--------\n[A] twin ...,0.0,[A] twin bed [S] positive\n--------\n[A] twin ...,0.0,<|aspect|> kamarnya <|sentiment|> negative\n--...,0.666667,<|aspect|> kamarnya <|sentiment|> negative\n--...,0.666667
3,3503,as,"over all baik , hanya sja akan lebih memuaskan...",[A] over all [S] positive\n--------\n[A] air h...,[A] air hot waternya [S] negative\n--------\n[...,1.000000,[A] air hot waternya [S] negative\n--------\n[...,1.000000,( air hot waternya | negative )\n--------\n( o...,1.000000,( air hot waternya | negative )\n--------\n( o...,1.000000,[A] over all [S] positive\n--------\n[A] air h...,1.0,[A] over all [S] positive\n--------\n[A] air h...,1.0,<|aspect|> air hot waternya <|sentiment|> nega...,1.000000,<|aspect|> air hot waternya <|sentiment|> nega...,1.000000
4,3504,as,fasilatas sesuia . [A] [S] =>,[A] fasilatas [S] positive,[A] fasilatas [S] negative,0.000000,[A] fasilitatas [S] negative,0.000000,( fasilatas | positive ),1.000000,( fasilitatas | positive ),0.000000,[A] fasilatas [S] positive,1.0,[A] fasilatas [S] positive,1.0,<|aspect|> fasilatas <|sentiment|> positive,1.000000,<|aspect|> fasilitatas <|sentiment|> positive,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1191,2847,as,ac nya gk dingin [A] [S] =>,[A] ac nya [S] negative,[A] ac nya [S] negative,1.000000,[A] ac nya [S] negative,1.000000,( ac nya | negative ),1.000000,( ac nya | negative ),1.000000,[A] ac nya [S] negative,1.0,[A] ac nya [S] negative,1.0,<|aspect|> ac nya <|sentiment|> negative,1.000000,<|aspect|> ac nya <|sentiment|> negative,1.000000
1192,2848,as,kamarnya enak luas tapi dapet sprei yg masih l...,[A] kamarnya [S] positive\n--------\n[A] sprei...,[A] kamarnya [S] positive\n--------\n[A] bau [...,0.800000,[A] kamarnya [S] positive\n--------\n[A] bau [...,0.800000,( kamarnya | positive )\n--------\n( sprei | n...,1.000000,( kamarnya | positive )\n--------\n( sprei | n...,1.000000,[S] positive [A] kamarnya\n--------\n[S] negat...,0.0,[S] positive [A] kamarnya\n--------\n[S] negat...,0.0,<|aspect|> sprei <|sentiment|> negative\n-----...,1.000000,<|aspect|> sprei <|sentiment|> negative\n-----...,1.000000
1193,2849,as,"kebersi

In [103]:
new_df[selected_columns].rename({'target_list': 'target_list', 'prediction_list': 'prediction_list-mvp_aos-constrained_decoding', 'f1_score': 'f1_score-mvp_aos-constrained_decoding'}, axis=1).to_csv('notebooks/error_analysis_hoasa_hotel_indo_seed123.csv', index=False)